# 07c — Train & Evaluate (Random-repeats branch)

Third parallel branch alongside `07_train_eval.ipynb` (spatial k-fold,
formal) and `07b_train_eval_bootstrap.ipynb` (bootstrap resampling with
replacement). Uses REPEATED RANDOM RE-SPLITS: each repeat splits the full
dataset by percentage, NO resampling with replacement, NO duplicates, NO
dedup needed -- every point used exactly once per repeat, full n retained
every time (unlike 07b, where bootstrap dedup drops ~35-40% of points per
repeat).

**Why this branch exists:** originally built to test whether 07b's
variance (val PR-AUC ranging ~0.59-0.74 for B_linear, with a persistent
val-vs-test gap) was an artifact of bootstrap's resample+dedup step. It's
since become the primary direction going forward, with three further
changes on top of that original motivation:

1. **60/20/20 split** (was 70/15/15), **stratified** by label so each of
   train/val/test preserves the dataset's natural positive/negative
   ratio (or a gently-adjusted one via `target_pos_frac` in
   `configs/eval_random_repeats.yaml`) instead of a plain shuffle letting
   the positive rate drift by chance across splits -- see
   `train._stratified_split`'s docstring.
2. **Adaptive decision threshold** instead of a fixed 0.5. A fixed 0.5
   threshold on an imbalanced crash dataset can be far too strict (all
   predicted probabilities pushed low by the class imbalance, so almost
   nothing crosses 0.5 and recall collapses) -- see
   `evaluate.find_optimal_threshold`'s docstring for the three methods
   (`f1`, `youden`, `cost_sensitive`) and how they trade precision for
   recall differently. Threshold is learned from val at the
   early-stopped epoch and frozen for test; `configs/eval_random_repeats.yaml`
   defaults to `cost_sensitive` at 10:1 (FN:FP), i.e. missing a
   crash-risk point is weighted like ten false alarms, reflecting a
   safety-critical framing where under-flagging is costlier than a false
   alarm. Every repeat's chosen threshold + method is logged alongside
   its metrics (`threshold_used`, `threshold_method`).
3. **100-epoch warmup + early stopping** (`warmup_epochs`, `patience` in
   config) -- was already supported by `train_one_fold`, now surfaced
   explicitly in `eval_random_repeats.yaml` rather than left to
   `config.get()` fallback defaults.

**Same significance-testing caveat as 07b applies here too:** no
Wilcoxon / Nadeau-Bengio tests run in this notebook. Those are built for
paired, correlated k-fold scores; repeated random re-splits aren't that
either. Descriptive aggregates (mean +/- std across repeats) only.

GPU recommended, not required (graphs are small).

In [ ]:
REPO_URL = "https://github.com/AditPradana36/crash-dualgraph.git"
REPO_DIR = "/content/crash-dualgraph"

import os
if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull

import sys
sys.path.append(f"{REPO_DIR}/src")

from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# TEMP: install locally patched train.py / models.py / plot_history.py
# until pushed to GitHub. Skip this cell once the repo itself is updated.
from google.colab import files
import shutil

print("Upload train.py, models.py, and plot_history.py:")
uploaded = files.upload()
for fname in uploaded:
    shutil.move(fname, f"{REPO_DIR}/src/{fname}")
print("Patched files installed:", list(uploaded.keys()))

In [ ]:
!pip install -q torch_geometric xgboost scikit-learn scipy pyyaml pandas tqdm

In [ ]:
import yaml
from pathlib import Path
import torch

with open(f"{REPO_DIR}/configs/paths.yaml") as f:
    paths_cfg = yaml.safe_load(f)
with open(f"{REPO_DIR}/configs/eval_random_repeats.yaml") as f:
    eval_cfg = yaml.safe_load(f)
with open(f"{REPO_DIR}/configs/model_random_repeats.yaml") as f:
    model_cfg = yaml.safe_load(f)

PROCESSED_DIR = Path(paths_cfg["processed_dir"])
OUTPUTS_DIR = Path(paths_cfg["outputs_dir"])
# separate checkpoint/metrics dirs from 07 and 07b -- keeps this branch's
# results from colliding with either of the other two
CHECKPOINT_DIR = OUTPUTS_DIR / "checkpoints_random_repeats"
METRICS_DIR = OUTPUTS_DIR / "metrics_random_repeats"
for d in [CHECKPOINT_DIR, METRICS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

device = "cuda" if torch.cuda.is_available() else "cpu"
config = {"batch_size": eval_cfg.get("batch_size") or 128,
          "epoch_cap": eval_cfg.get("epoch_cap") or 400,
          "warmup_epochs": eval_cfg.get("warmup_epochs") or 100,
          "patience": eval_cfg.get("patience") or 40,
          "lr_patience": eval_cfg.get("lr_patience") or 10,
          "lr": eval_cfg.get("lr") or 5e-3,
          "weight_decay": eval_cfg.get("weight_decay") or 1e-4,
          "fusion_dim": model_cfg.get("fusion_dim") or 128,
          # 60/20/20: val_frac + test_frac each 0.20, remaining 0.60 is train
          "val_frac": eval_cfg.get("val_frac") or 0.20,
          "test_frac": eval_cfg.get("test_frac") or 0.20,
          # stratified split: preserves (or gently nudges, via
          # target_pos_frac) the positive rate per split instead of a
          # plain shuffle drifting it by chance -- see
          # train._stratified_split's docstring
          "label_col": eval_cfg.get("label_col") or "label",
          "target_pos_frac": eval_cfg.get("target_pos_frac"),
          # adaptive decision threshold, learned on val and frozen for
          # test -- see evaluate.find_optimal_threshold's docstring.
          # "fixed" reproduces the old hardcoded-0.5 behavior.
          "threshold_method": eval_cfg.get("threshold_method") or "fixed",
          "threshold_fn_cost": eval_cfg.get("threshold_fn_cost") or 10.0,
          "threshold_fp_cost": eval_cfg.get("threshold_fp_cost") or 1.0,
          "num_workers": eval_cfg.get("num_workers") or 0,
          "use_amp": eval_cfg.get("use_amp", True)}
N_REPEATS = eval_cfg.get("n_repeats") or 20
print(f"Device: {device} | n_repeats: {N_REPEATS}")
print(f"Split: {1 - config['val_frac'] - config['test_frac']:.0%}/{config['val_frac']:.0%}/{config['test_frac']:.0%} "
      f"(train/val/test), stratified by '{config['label_col']}', target_pos_frac={config['target_pos_frac']}")
print(f"Threshold: {config['threshold_method']}"
      + (f" (fn_cost={config['threshold_fn_cost']}, fp_cost={config['threshold_fp_cost']})"
         if config['threshold_method'] == "cost_sensitive" else ""))
print(f"Warmup: {config['warmup_epochs']} epochs, patience: {config['patience']}, epoch_cap: {config['epoch_cap']}")
print(f"Full config: {config}")


In [ ]:
import pandas as pd
import graph_datasets as ds
import train as tr
import evaluate as ev
import models

index_df = pd.read_parquet(PROCESSED_DIR / "dataset_index.parquet")
dataset = ds.DualGraphDataset(index_df, PROCESSED_DIR / "svg_graphs", PROCESSED_DIR / "tvg_graphs")
print(f"Dataset: {len(dataset)} points (random-repeats branch -- no fold_cols, no bootstrap resampling)")

svg_kwargs = dict(hidden_dim=model_cfg.get("hidden_dim", 128), heads=model_cfg.get("heads", 4),
                   num_layers=model_cfg.get("svg_layers", 2), dropout=model_cfg.get("dropout", 0.45),
                   signage_vocab=5, light_pole_vocab=4, road_marking_vocab=2, cat_embed_dim=2)
tvg_kwargs = dict(hidden_dim=model_cfg.get("hidden_dim", 128), heads=model_cfg.get("heads", 4),
                   num_layers=model_cfg.get("tvg_layers", 2), dropout=model_cfg.get("dropout", 0.45),
                   building_type_vocab=58, highway_vocab=13,
                   building_type_embed_dim=8, highway_embed_dim=4)

In [ ]:
# ── Train every primary scenario x head depth ─────────────────────────
# Split into one cell per scenario below, same rationale as 07/07b: run /
# monitor / interrupt independently rather than one long nested loop.
PRIMARY_SCENARIOS = ["A", "B", "C", "D", "E"]
HEAD_DEPTHS = ["linear", "mlp2"]
all_results = {}

### Scenario A — SVG only

In [ ]:
# ── Scenario A: linear + mlp2 ─────────────────────────────────
for depth in HEAD_DEPTHS:
    key = f"A_{depth}"
    print(f"\n=== {key} ===")
    results = tr.run_scenario_random_repeats("A", depth, use_ablation=False, dataset=dataset,
                                               n_repeats=N_REPEATS, config=config, svg_kwargs=svg_kwargs,
                                               tvg_kwargs=tvg_kwargs, device=device, checkpoint_dir=CHECKPOINT_DIR)
    all_results[key] = results
    print(f"  {len(results)} repeat-runs complete.")

### Scenario B — TVG only

In [ ]:
# ── Scenario B: linear + mlp2 ─────────────────────────────────
for depth in HEAD_DEPTHS:
    key = f"B_{depth}"
    print(f"\n=== {key} ===")
    results = tr.run_scenario_random_repeats("B", depth, use_ablation=False, dataset=dataset,
                                               n_repeats=N_REPEATS, config=config, svg_kwargs=svg_kwargs,
                                               tvg_kwargs=tvg_kwargs, device=device, checkpoint_dir=CHECKPOINT_DIR)
    all_results[key] = results
    print(f"  {len(results)} repeat-runs complete.")

### Scenario C — dual graph (concat)

In [ ]:
# ── Scenario C: linear + mlp2 ─────────────────────────────────
for depth in HEAD_DEPTHS:
    key = f"C_{depth}"
    print(f"\n=== {key} ===")
    results = tr.run_scenario_random_repeats("C", depth, use_ablation=False, dataset=dataset,
                                               n_repeats=N_REPEATS, config=config, svg_kwargs=svg_kwargs,
                                               tvg_kwargs=tvg_kwargs, device=device, checkpoint_dir=CHECKPOINT_DIR)
    all_results[key] = results
    print(f"  {len(results)} repeat-runs complete.")

### Scenario D — dual graph (late fusion)

In [ ]:
# ── Scenario D: linear + mlp2 ─────────────────────────────────
for depth in HEAD_DEPTHS:
    key = f"D_{depth}"
    print(f"\n=== {key} ===")
    results = tr.run_scenario_random_repeats("D", depth, use_ablation=False, dataset=dataset,
                                               n_repeats=N_REPEATS, config=config, svg_kwargs=svg_kwargs,
                                               tvg_kwargs=tvg_kwargs, device=device, checkpoint_dir=CHECKPOINT_DIR)
    all_results[key] = results
    print(f"  {len(results)} repeat-runs complete.")

### Scenario E — dual graph (cross-attention)

In [ ]:
# ── Scenario E: linear + mlp2 ─────────────────────────────────
for depth in HEAD_DEPTHS:
    key = f"E_{depth}"
    print(f"\n=== {key} ===")
    results = tr.run_scenario_random_repeats("E", depth, use_ablation=False, dataset=dataset,
                                               n_repeats=N_REPEATS, config=config, svg_kwargs=svg_kwargs,
                                               tvg_kwargs=tvg_kwargs, device=device, checkpoint_dir=CHECKPOINT_DIR)
    all_results[key] = results
    print(f"  {len(results)} repeat-runs complete.")

In [ ]:
# ── Ablation: B-E only (F deferred) ────────────────────────────────────
# Split into one cell per scenario, same rationale as the primary block above.

### Ablation B+

In [ ]:
# ── Scenario B ablation: linear + mlp2 ─────────────────────────────────
for depth in HEAD_DEPTHS:
    key = f"B_{depth}_ablation"
    print(f"\n=== {key} ===")
    results = tr.run_scenario_random_repeats("B", depth, use_ablation=True, dataset=dataset,
                                               n_repeats=N_REPEATS, config=config, svg_kwargs=svg_kwargs,
                                               tvg_kwargs=tvg_kwargs, device=device, checkpoint_dir=CHECKPOINT_DIR)
    all_results[key] = results
    print(f"  {len(results)} repeat-runs complete.")

### Ablation C+

In [ ]:
# ── Scenario C ablation: linear + mlp2 ─────────────────────────────────
for depth in HEAD_DEPTHS:
    key = f"C_{depth}_ablation"
    print(f"\n=== {key} ===")
    results = tr.run_scenario_random_repeats("C", depth, use_ablation=True, dataset=dataset,
                                               n_repeats=N_REPEATS, config=config, svg_kwargs=svg_kwargs,
                                               tvg_kwargs=tvg_kwargs, device=device, checkpoint_dir=CHECKPOINT_DIR)
    all_results[key] = results
    print(f"  {len(results)} repeat-runs complete.")

### Ablation D+

In [ ]:
# ── Scenario D ablation: linear + mlp2 ─────────────────────────────────
for depth in HEAD_DEPTHS:
    key = f"D_{depth}_ablation"
    print(f"\n=== {key} ===")
    results = tr.run_scenario_random_repeats("D", depth, use_ablation=True, dataset=dataset,
                                               n_repeats=N_REPEATS, config=config, svg_kwargs=svg_kwargs,
                                               tvg_kwargs=tvg_kwargs, device=device, checkpoint_dir=CHECKPOINT_DIR)
    all_results[key] = results
    print(f"  {len(results)} repeat-runs complete.")

### Ablation E+

In [ ]:
# ── Scenario E ablation: linear + mlp2 ─────────────────────────────────
for depth in HEAD_DEPTHS:
    key = f"E_{depth}_ablation"
    print(f"\n=== {key} ===")
    results = tr.run_scenario_random_repeats("E", depth, use_ablation=True, dataset=dataset,
                                               n_repeats=N_REPEATS, config=config, svg_kwargs=svg_kwargs,
                                               tvg_kwargs=tvg_kwargs, device=device, checkpoint_dir=CHECKPOINT_DIR)
    all_results[key] = results
    print(f"  {len(results)} repeat-runs complete.")

In [ ]:
# ── Scenario G: XGBoost, separate path (random-repeats version) ────────
import baseline_features
from xgboost import XGBClassifier

feat_table = baseline_features.build_feature_table(index_df["point_id"].tolist(),
                                                     PROCESSED_DIR / "svg_graphs", PROCESSED_DIR / "tvg_graphs", torch)
feat_table = feat_table.merge(index_df[["point_id", "label"]], on="point_id")
feature_cols = [c for c in feat_table.columns if c not in ["point_id", "label"]]

n_total = len(feat_table)
g_results = []
for repeat in range(N_REPEATS):
    repeat_seed = 42 + repeat
    # same stratified-split scheme as A-E in this branch (see
    # train._stratified_split), so G is evaluated on a like-for-like
    # basis rather than a plain shuffle
    train_, val_, test = tr._stratified_split(
        feat_table, config["label_col"], config["val_frac"], config["test_frac"],
        repeat_seed, config.get("target_pos_frac"))

    clf = XGBClassifier(n_estimators=200, max_depth=4, eval_metric="aucpr", random_state=42)
    clf.fit(train_[feature_cols], train_["label"])

    # adaptive threshold: fit on val predictions, freeze for test -- same
    # contract as train_one_fold's threshold handling for A-E
    if config["threshold_method"] == "fixed":
        chosen_threshold = 0.5
        threshold_method_used = "fixed"
    else:
        val_prob = clf.predict_proba(val_[feature_cols])[:, 1]
        chosen_threshold, threshold_method_used, _ = ev.find_optimal_threshold(
            val_["label"].values, val_prob, method=config["threshold_method"],
            fn_cost=config["threshold_fn_cost"], fp_cost=config["threshold_fp_cost"])

    prob = clf.predict_proba(test[feature_cols])[:, 1]
    metrics = ev.compute_metrics(test["label"].values, prob, threshold=chosen_threshold)
    metrics["threshold_method"] = threshold_method_used
    g_results.append({"repeat": repeat, "n_train": len(train_), "n_val": len(val_), "n_test": len(test),
                       **metrics})

all_results["G"] = g_results
print(f"Scenario G: {len(g_results)} repeat-runs complete.")


In [ ]:
# ── Aggregate + report EVERY scenario, regardless of performance ────────
agg_rows = []
for key, results in all_results.items():
    agg = ev.aggregate_fold_results(results)   # works identically regardless of split scheme
    row = {"scenario": key}
    for metric, (mean, std) in agg.items():
        row[f"{metric}_mean"] = mean
        row[f"{metric}_std"] = std
    agg_rows.append(row)

agg_df = pd.DataFrame(agg_rows)
agg_df.to_csv(METRICS_DIR / "all_scenarios_summary_random_repeats.csv", index=False)
display(agg_df)

## Threshold diagnostics

Per-scenario mean/std of the threshold each repeat actually chose
(`threshold_used`), plus a count of which method fired per repeat
(`threshold_method`) -- mainly to catch `fallback_no_signal`, which
means a repeat's val split happened to contain only one class (can
happen on small/unlucky splits) and fell back to the fixed 0.5 default
rather than actually optimizing anything for that repeat.

In [ ]:
threshold_rows = []
for key, results in all_results.items():
    method_counts = ev.summarize_categorical_field(results, "threshold_method")
    thresh_mean, thresh_std = ev.aggregate_fold_results(results).get("threshold_used", (float("nan"), float("nan")))
    threshold_rows.append({"scenario": key, "threshold_mean": thresh_mean, "threshold_std": thresh_std,
                            "methods_used": method_counts})

threshold_df = pd.DataFrame(threshold_rows)
threshold_df.to_csv(METRICS_DIR / "threshold_diagnostics_random_repeats.csv", index=False)
display(threshold_df)


In [ ]:
# ── Head-depth comparison: descriptive only (no formal significance test
#    here -- see notebook intro for why). ──
depth_compare = []
for scenario in PRIMARY_SCENARIOS:
    lin = agg_df[agg_df["scenario"] == f"{scenario}_linear"]["pr_auc_mean"].iloc[0]
    mlp = agg_df[agg_df["scenario"] == f"{scenario}_mlp2"]["pr_auc_mean"].iloc[0]
    depth_compare.append({"scenario": scenario, "linear_pr_auc": lin, "mlp2_pr_auc": mlp})

depth_df = pd.DataFrame(depth_compare)
display(depth_df)

WINNING_DEPTH = "linear" if depth_df["linear_pr_auc"].mean() >= depth_df["mlp2_pr_auc"].mean() else "mlp2"
print(f"\nWinning head depth (by mean PR-AUC across scenarios): {WINNING_DEPTH}")
print("Descriptive only -- no Wilcoxon/Nadeau-Bengio run in this branch (see intro).")

## Comparing against 07b (bootstrap)

**Note:** since 07c now uses 60/20/20 splits and 07b still uses
70/15/15 (`configs/eval_bootstrap.yaml`, untouched), this is no longer a
strict like-for-like comparison the way it was when the two branches
shared identical split percentages -- treat differences here as
suggestive, not a controlled ablation. It's still useful for checking
whether the broad variance/gap pattern from 07b persists, just keep the
split-ratio difference in mind when reading it.

Loads 07b's `all_scenarios_summary_bootstrap.csv` alongside this
branch's summary and compares std (spread) and mean PR-AUC directly for
matching scenario/depth keys.

In [ ]:
bootstrap_summary_path = OUTPUTS_DIR / "metrics_bootstrap" / "all_scenarios_summary_bootstrap.csv"
if bootstrap_summary_path.exists():
    boot_df = pd.read_csv(bootstrap_summary_path)
    compare = agg_df[["scenario", "pr_auc_mean", "pr_auc_std"]].merge(
        boot_df[["scenario", "pr_auc_mean", "pr_auc_std"]],
        on="scenario", suffixes=("_random_repeats", "_bootstrap"))
    display(compare)
else:
    print("07b's summary CSV not found yet -- run 07b first to compare.")

## Epoch-level diagnostics

Per-repeat training history is saved under
`CHECKPOINT_DIR/{tag}_history/repeat{N}.json`, same JSON shape as 07b's.
Same note as 07b: `plot_history.py`'s existing helpers assume
`{fold_col}_fold{id}` naming, not `repeat{N}` -- using the same direct
JSON-loading fallback as 07b rather than assuming an unverified helper
exists.

In [ ]:
import json
import matplotlib.pyplot as plt

history_path = CHECKPOINT_DIR / "A_linear_history" / "repeat0.json"
history = json.loads(history_path.read_text())

epochs = [h["epoch"] for h in history]
best_epoch = max(range(len(history)), key=lambda i: history[i]["val_pr_auc"])

fig, ax1 = plt.subplots(figsize=(8, 4))
ax1.plot(epochs, [h["train_loss"] for h in history], label="train_loss", color="tab:blue")
ax1.set_xlabel("epoch"); ax1.set_ylabel("train_loss", color="tab:blue")

ax2 = ax1.twinx()
ax2.plot(epochs, [h["val_pr_auc"] for h in history], label="val_pr_auc", color="tab:orange")
ax2.plot(epochs, [h["val_auroc"] for h in history], label="val_auroc", color="tab:green")
ax2.axvline(best_epoch, color="gray", linestyle="--", label=f"best epoch ({best_epoch})")
ax2.set_ylabel("val metric")

fig.legend(loc="upper right", bbox_to_anchor=(0.9, 0.9))
plt.title(f"repeat0 history ({history_path.name})")
plt.tight_layout()
plt.show()

In [ ]:
print("Random-repeats branch complete.")
print("Compare mean/std against 07 (k-fold) and 07b (bootstrap) to see whether")
print("variance/gap patterns persist without bootstrap's resample+dedup step.")